In [1]:
from dotenv import load_dotenv
import os
from pathlib import Path
load_dotenv()
PATH_FINTOC_2022 = Path(os.getenv('PATH_FINTOC_2022'))

In [2]:
FINTOC_EN = PATH_FINTOC_2022/'data'/'en'
FINTOC_EN_PDF =FINTOC_EN/'pdf'
FINTOC_EN_ANNOTS = FINTOC_EN/'annots'
FINTOC_EN_PRED = FINTOC_EN/'preds'

JSON_EXTENSION = "fintoc4.json"

In [3]:
if not FINTOC_EN_PRED.exists():
    FINTOC_EN_PRED.mkdir()
FILE_PATHS = list(FINTOC_EN_PDF.iterdir())
FILE_PATHS.sort()
FILE_PATHS = FILE_PATHS[:10]

In [4]:
import json

import pager
from pager.pager_pipe_line import pdf2region_row

from pager.doc_model import MinerPDFModel
from pager.doc_model import LogicTreeModel
from pager.doc_model.converters import PDF2LogicTree
from pager.doc_model.dtype import Document, Page
from pager import Region

from typing import Dict

pdf_reader = MinerPDFModel({"page_model": pdf2region_row})
pdf2tree = PDF2LogicTree()
tree = LogicTreeModel()

In [5]:
file_path = FILE_PATHS[0]
pdf_reader.read_from_file(file_path)
pdf_reader.extract()
pdf2tree.convert(pdf_reader, tree)
file_path

/home/daniil/project/PageR/env/lib/python3.12/site-packages/torch_geometric/nn/conv/tag_conv.py:102: UserWarning: Converting sparse tensor to CSR format for more efficient processing. Consider converting your sparse tensor to CSR format beforehand to avoid repeated conversion (got 'torch.sparse_coo')
  return spmm(adj_t, x, reduce=self.aggr)
/home/daniil/project/PageR/env/lib/python3.12/site-packages/torch_geometric/utils/_spmm.py:71: UserWarning: Sparse CSR tensor support is in beta state. If you miss a functionality in the sparse tensor support, please submit a feature request to https://github.com/pytorch/pytorch/issues. (Triggered internally at /pytorch/aten/src/ATen/SparseCsrTensorImpl.cpp:53.)
  src = src.to_sparse_csr()
/home/daniil/project/PageR/env/lib/python3.12/site-packages/torch/nn/modules/module.py:1775: UserWarning: Implicit dimension choice for softmax has been deprecated. Change the call to include dim=X as an argument.
  return self._call_impl(*args, **kwargs)


PosixPath('/home/dataset/fintoc2022/data/en/pdf/1.9.900555.pdf')

# Отладка на одном документе

In [6]:
doc_regions = [region for page in tree.document.pages for region in page.regions]
regions_num_page = [page.num_page for page in tree.document.pages for _ in page.regions]

In [7]:
# for region in doc_regions:
#     print(region.md)

### Кластер регионов

In [9]:
_font_from_doc = {}
_list_font_type = []
def get_vec_feature(row):
    fd = row.font.to_dict()
    name = fd['name']
    size = int(round(fd['size']/2))
    width = int(round(fd['width']/0.1))
    italic = int(round(fd['italic']/0.1))
    return (name, size, width, italic)

def get_id_exist_cluster(feature):
    _list_font_type.append(feature)
    if feature in _font_from_doc.keys():
         return _font_from_doc[feature] 
    return None
        
def add_new_cluster(feature):
    indexs= list(_font_from_doc.values())
    index = max(indexs)+1 if len(indexs) != 0 else 0
    _font_from_doc[feature] = index
    clusters[index] = []
    return index
    
clusters = {}
for region in doc_regions:
    feature_row = get_vec_feature(region)
    index = get_id_exist_cluster(feature_row)
    if index is None:
        index = add_new_cluster(feature_row)
    clusters[index].append(region)

### Поиск кластера основного текста

In [10]:
max_clust = -1
max_size = 0
for ind, regions in clusters.items():
    count_word = len([w for reg in regions for w in reg.words])
    if max_size < count_word:
        max_clust = ind
        max_size = count_word
font_text =  clusters[max_clust][0].rows[0].font
print(max_size)
font_text.to_dict()

5392


{'name': 'EPEAMU+NotoSans',
 'width': 0.0,
 'italic': 0.0,
 'size': 8.499899999999911}

### Поиск кластеров заголовков

In [11]:
header_cluster = {ind: cl[0] for ind, cl in clusters.items() if cl[0].font > font_text}

### Сортировка кластеров заголовков

In [12]:
header_cluster_index = list(header_cluster.keys())
header_cluster_index.sort(key=lambda ind: header_cluster[ind].font, reverse=True)
for header_ind in header_cluster_index:
    print(get_vec_feature(header_cluster[header_ind]))

level_header_cluster = {cl:level+1 for level, cl in enumerate(header_cluster_index)}
# level_header_cluster

('TKRTDS+SchrodersCircular-Bold', 14, 10, 0)
('TKRTDS+SchrodersCircular-Bold', 11, 10, 0)
('TKRTDS+SchrodersCircular-Bold', 12, 10, 0)
('EPEAMU+NotoSans', 12, 2, 0)
('EPEAMU+NotoSans', 12, 0, 0)
('EPEAMU+NotoSans', 8, 0, 0)
('TKRTDS+SchrodersCircular-Bold', 6, 10, 0)
('QHVYUR+NotoSans-Bold', 6, 10, 0)
('EPEAMU+NotoSans', 6, 5, 0)
('EPEAMU+NotoSans', 6, 3, 0)
('EPEAMU+NotoSans', 6, 1, 0)
('XFMZLQ+SchrodersCircular-Book', 6, 0, 0)
('TKRTDS+SchrodersCircular-Bold', 5, 10, 0)
('QHVYUR+NotoSans-Bold', 5, 10, 0)
('XFMZLQ+SchrodersCircular-Book', 8, 0, 0)
('EPEAMU+NotoSans', 5, 2, 0)
('EPEAMU+NotoSans', 5, 1, 0)
('EPEAMU+NotoSans', 5, 5, 0)
('TKRTDS+SchrodersCircular-Bold', 5, 7, 0)
('EPEAMU+NotoSans', 5, 0, 0)
('EPEAMU+NotoSans', 6, 0, 0)
('EPEAMU+NotoSans', 6, 2, 0)
('QHVYUR+NotoSans-Bold', 4, 10, 0)
('TKRTDS+SchrodersCircular-Bold', 4, 10, 0)


### Создание регионов

In [13]:
num_cluster = [ _font_from_doc[vec] for vec in _list_font_type]

def seg_region(region, cl, num_page):
    
    if cl in level_header_cluster:
        region.header_level = level_header_cluster[cl]
        region.label = 'header'
    else:
        region.label = 'other'
    return region

page_regions = dict()
for region, cl, num_page in zip(doc_regions, num_cluster, regions_num_page):
    seg_region(region, cl, num_page)
    if num_page in page_regions:
        page_regions[num_page].append(region)
    else:
        page_regions[num_page] = [region]

### Построение дерева (Переиспользование класса конвертера)

In [14]:


class CustomJson2LogicTree(PDF2LogicTree):
    def upload(self, regions, output_model:LogicTreeModel):
        reg_pages = [(n, reg) for n, reg in  page_regions.items()]
        reg_pages.sort(key= lambda x: x[0])
        document = Document([Page(regions=regs, num_page=num) for num, regs in (reg_pages)])
        regions = [reg for page in document.pages for reg in page.regions]
        tree = self.get_tree_from_doc(regions)
        self.set_level(regions, tree['parent'])
        
        output_model.document = document
        output_model.regions = regions
        output_model.edges = tree


creater = CustomJson2LogicTree()
tree = LogicTreeModel()
creater.upload(regions, tree)

In [15]:
print(tree.document.md)

# (a Luxembourg domiciled open-ended investment company) 

# Schroder GAIA 

# Prospectus 

## August 2019 

# United Kingdom 


<!-- page_num: 0 --> 
---Schroder GAIA

(a Luxembourg domiciled open-ended investment company)

# Prospectus 

## August 2019 

Schroder Investment Management (Europe) S.A.

Internet Site: www.schroders.lu


<!-- page_num: 2 --> 
---## Important Information 

Copies of this Prospectus can be obtained from and enquiries regarding the Company should be addressed to:

Schroder Investment Management (Europe) S.A. 5, rue Höhenhof 1736 Senningerberg Grand Duchy of Luxembourg Tel: (+352) 341 342 202 Fax: (+352) 341 342 342

This prospectus (the "Prospectus") should be read in its

entirety before making any application for Shares. If you are in any doubt about the contents of this Prospectus you should consult your financial or other professional adviser. Shares are offered on the basis of the information contained

in this Prospectus and the documents referred to h

# Упаковка в один класс

In [8]:
class CustomJson2LogicTree(PDF2LogicTree):
    def upload(self, reg_page, output_model:LogicTreeModel):
        reg_page = [(n, reg) for n, reg in  reg_page.items()]
        reg_page.sort(key= lambda x: x[0])
        reg_page = [regs for n, regs in reg_page]
        document = Document([Page(regions=regs, num_page=num) for num, regs in enumerate(reg_page)])
        regions = [reg for page in document.pages for reg in page.regions]
        tree = self.get_tree_from_doc(regions)
        self.set_level(regions, tree['parent'])
        
        output_model.document = document
        output_model.regions = regions
        output_model.edges = tree

class ExtractTree:
    def __init__(self):
        self.creater = CustomJson2LogicTree()
        self._font_from_doc = {}
        self._list_font_type = []

        
    def extract(self, tree:LogicTreeModel):
        doc_regions = [region for page in tree.document.pages for region in page.regions]
        region_num_page = [page.num_page for page in tree.document.pages for _ in page.regions]
        clusters = self.get_cluster(doc_regions)
        level_header_cluster = self.get_level_header(clusters)
        page_regions = self.get_regions(doc_regions, region_num_page, level_header_cluster)
        self.creater.upload(page_regions, tree)


    def get_regions(self, doc_regions, regions_num_page, level_header_cluster):
        num_cluster = [self._font_from_doc[vec] for vec in self._list_font_type]

        def seg_region(region, cl, num_page):
            
            if cl in level_header_cluster:
                region.header_level = level_header_cluster[cl]
                region.label = 'header'
            else:
                region.label = 'other'
            return region
        
        page_regions = dict()
        for region, cl, num_page in zip(doc_regions, num_cluster, regions_num_page):
            seg_region(region, cl, num_page)
            if num_page in page_regions:
                page_regions[num_page].append(region)
            else:
                page_regions[num_page] = [region]

        return page_regions

        
    def get_level_header(self, clusters):
        main_clust, main_font = self.get_main_cluster(clusters)
        header_cluster = {ind: cl[0] for ind, cl in clusters.items() if cl[0].font > main_font}
        header_cluster_index = list(header_cluster.keys())
        header_cluster_index.sort(key=lambda ind: header_cluster[ind].font, reverse=True)        
        level_header_cluster = {cl:level+1 for level, cl in enumerate(header_cluster_index)}
        return level_header_cluster
        
    def get_main_cluster(self, clusters):
        def get_words(reg):
            words = []
            for row in reg.rows:
                words.extend(row.words)
            return words
        max_clust = -1
        max_size = 0
        for ind, regions in clusters.items():
            
            count_word = len([w for reg in regions for w in get_words(reg)])
            if max_size < count_word:
                max_clust = ind
                max_size = count_word
        font_text =  clusters[max_clust][0].rows[0].font
        return max_clust, font_text

    def get_cluster(self, doc_regions):
        self._font_from_doc = {}
        self._list_font_type = []
        def get_vec_feature(row):
            fd = row.font.to_dict()
            name = fd['name']
            size = int(round(fd['size']/2))
            width = int(round(fd['width']/0.1))
            italic = int(round(fd['italic']/0.1))
            return (name, size, width, italic)
        
        def get_id_exist_cluster(feature):
            self._list_font_type.append(feature)
            if feature in self._font_from_doc.keys():
                 return self._font_from_doc[feature] 
            return None
                
        def add_new_cluster(feature):
            indexs= list(self._font_from_doc.values())
            index = max(indexs)+1 if len(indexs) != 0 else 0
            self._font_from_doc[feature] = index
            clusters[index] = []
            return index
            
        clusters = {}
        for region in doc_regions:
            feature_row = get_vec_feature(region)
            index = get_id_exist_cluster(feature_row)
            if index is None:
                index = add_new_cluster(feature_row)
            clusters[index].append(region)
        return clusters
        
       
     
   

extract_tree=ExtractTree() 

In [9]:
def tree2fintoc_json(doc_tree) -> Dict:
    fintoc_json = []
    for id_node, node in doc_tree['nodes']['regions'].items():
        if node['label'] == 'header':
            reg = {
                "text":node['text'],
                "id": id_node,
                "page": node['metainfo']['page'] + 1, # В PageR специально оставил нумерацию с нуля (поскольку номер определяется не порядком, а пишится на странице и это еще не реализовано)
                "depth": node['header_level']
            }
            fintoc_json.append(reg)
    return fintoc_json

N = len(FILE_PATHS)
for i, file_path in enumerate(FILE_PATHS):    
    pdf_reader.read_from_file(file_path)
    pdf_reader.extract()
    pdf2tree.convert(pdf_reader, tree)
    extract_tree.extract(tree)
    
    doc_tree = tree.to_dict()
    pred_path = FINTOC_EN_PRED/(file_path.name+'.'+JSON_EXTENSION)
    # print(tree.document.md)
    
    rez = tree2fintoc_json(doc_tree)
    with open(pred_path, 'w') as f:
        json.dump(rez, f)

    print(f'{i+1}/{N} ({(i+1)/N*100:.2f}%)')

1/10 (10.00%)
2/10 (20.00%)
3/10 (30.00%)
4/10 (40.00%)
5/10 (50.00%)
6/10 (60.00%)
7/10 (70.00%)
8/10 (80.00%)
9/10 (90.00%)
10/10 (100.00%)


In [10]:
import subprocess
import pandas as pd

In [11]:
subprocess.run(['python', 'metric.py', '--gt_folder', FINTOC_EN_ANNOTS, '--submission_folder', FINTOC_EN_PRED])
df = pd.read_csv('td_report.csv', delimiter='\t')
df[-2:-1][['Prec', 'Rec', 'F1']]

,Prec,Rec,F1
189,0.5559832458911166,0.5127790935638992,0.5190581039800979


In [12]:
df = pd.read_csv('toc_report.csv', delimiter='\t')
tbl = df[-4:-3]
tbl[['Inex08-P', 'Inex08-R', 'Inex08-F1', 'Inex08-Title acc', 'Inex08-Level acc']]

,Inex08-P,Inex08-R,Inex08-F1,Inex08-Title acc,Inex08-Level acc
215,31.1,25.5,27.3,40.3,19.2


In [70]:
df = pd.read_csv('toc_report.csv', delimiter='\t')
tbl = df[-4:-3]
tbl[['Inex08-P', 'Inex08-R', 'Inex08-F1', 'Inex08-Title acc', 'Inex08-Level acc']]

,Inex08-P,Inex08-R,Inex08-F1,Inex08-Title acc,Inex08-Level acc
170,26.8,23.8,24.8,36.6,13.3
